In [ ]:
!pip install pymongo

In [ ]:
from pymongo import MongoClient

## Establish a connection to the MongoDB Atlas cluster using the connection string.

In [ ]:
client = MongoClient(os.getenv("MONGODB_URI"))
db = client["projectdb"]

# Import Libraries

In [ ]:
import psycopg2
from psycopg2 import sql
import os
from pyjstat import pyjstat

In [ ]:
import requests
import json
from sqlalchemy import create_engine
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px 
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ETL (Extract, Transform, Load)

This script extracts data from Eurostat API, cleans it in MongoDB, and loads it into PostgreSQL for circular economy analysis.

## Extraction (API) and Loading (MongoDB)

In [ ]:
# Define the MongoDB collections where raw data will be stored.
# We separate the data into two collections based on the dataset timeframes.
col_old_raw = db["ewaste_2005_2017_rawflat"]
col_new_raw = db["ewaste_2018_2023_rawflat"]

# Map the API dataset codes to their respective MongoDB collections.
datasets = {
    "env_waselee": col_old_raw,
    "env_waseleeos": col_new_raw
}

# Iterate through each dataset to fetch and store data.
for dataset_code, collection in datasets.items():
    print(f"\n⬇️ Extracting dataset: {dataset_code}")

# EXTRACT Data From the Eurostat API in JSON format
    url = f"https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/{dataset_code}?format=JSON&lang=en"
    r = requests.get(url)
    r.raise_for_status()

# Parse JSON-stat → flat table
    dataset = pyjstat.Dataset.read(json.dumps(r.json()))
    df = dataset.write("dataframe")

    print(f"➡️ Extracted rows: {len(df)}")

# LOAD raw data into MongoDB
    records = df.to_dict(orient="records")
    collection.delete_many({})
    collection.insert_many(records)

    print(f"💾 Loaded {len(records)} raw flat documents into {collection.name}")

print("\n🎉 E (Extract) + L (Load) completed.")

# Transformation within MongoDB

In [ ]:
# Define an aggregation pipeline to standardize field names.

pipeline = [
    {
        "$set": {
            "country_code": "$geo",
            "ewaste_category": "$waste",
            "management_operation": "$wst_oper",
            "year": "$time",
            "unit": "$unit",
            "value": "$value"
        }
    },
    {
        # Remove the old field names to keep the document clean
        "$unset": ["geo", "waste", "wst_oper", "time"]
    }
]

# Apply the transformation to both collections.
db.ewaste_2005_2017_rawflat.update_many({}, pipeline)
db.ewaste_2018_2023_rawflat.update_many({}, pipeline)

In [ ]:
# Remove records where the 'value' is null/None, as they are not useful for analysis.
db.ewaste_2005_2017_rawflat.delete_many({"value": None})
db.ewaste_2018_2023_rawflat.delete_many({"value": None})

In [ ]:
# Convert the 'year' field from a String to an Integer type for correct sorting/plotting
db.ewaste_2005_2017_rawflat.update_many(
    {}, [{ "$set": { "year": { "$toInt": "$year" } } }]
)

# Loading into Pandas & PostgreSQL

In [ ]:
# Fetch the cleaned data back from MongoDB. Excluding '_id' as it's an internal Mongo field.
data_2005_2017 = list(col_old_raw.find({}, {'_id': 0})) # exclude '_id'
print(f"Retrieved {len(data_2005_2017)} records from 2005-2017 collection.")

# Retrieve data from the '2018_2023' collection
data_2018_2023 = list(col_new_raw.find({}, {'_id': 0})) # exclude '_id'
print(f"Retrieved {len(data_2018_2023)} records from 2018-2023 collection.")

In [ ]:
import pandas as pd

In [ ]:
# Merge the two time periods into a single DataFrame.
df = pd.DataFrame(data_2005_2017 + data_2018_2023)
print(f"Merged DataFrame created with {len(df)} rows.")

## DATA CLEANING AND TRANSFORMATION

In [ ]:
# Ensure data types are correct for analysis.
# Convert 'year' to integer type
df['Time'] = pd.to_numeric(df['Time'], errors='coerce', downcast='integer')

In [ ]:
# Convert 'value' to numeric (if not already)
df['value'] = pd.to_numeric(df['value'], errors='coerce')

In [ ]:
print(f"Cleaned DataFrame shape: {df.shape}")
print(df.head())

In [ ]:
# Filter years for analysis to 2012-2023 based on WEEE Directive revision
ewaste_2012_2023 = df[(df["Time"] >= 2012) & (df["Time"] <= 2023)]
 
print("Filtered dataset shape (2012–2023):", ewaste_2012_2023.shape)
print(ewaste_2012_2023.head())

In [ ]:
ewaste_2012_2023.info()

# CONNECT DATAFRAME TO POSTGRESQL

In [ ]:
#establishing the connection
conn = psycopg2.connect(database=os.getenv("POSTGRES_DB", "postgres"), user=os.getenv("POSTGRES_USER", "postgres"), password=os.getenv("POSTGRES_PASSWORD"), host=os.getenv("POSTGRES_HOST", "127.0.0.1"), port=os.getenv("POSTGRES_PORT", "5432"))

#Creating a cursor object using the cursor() method
cursor = conn.cursor()

#Executing an MYSQL function using the execute() method
cursor.execute("select version()")

# Fetch a single row using fetchone() method.
data = cursor.fetchone()
print("Connection established to: ",data)


In [ ]:
#Define Database Connection
DB_USER = os.getenv("POSTGRES_USER", "postgres")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_HOST = "localhost"     # Your host address
DB_PORT = "5432"          # Your port number
DB_NAME = "ewaste_data"

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
df.to_sql('ewaste_2012_2023', engine, if_exists='replace', index=False)
print("Data stored in PostgreSQL successfully!")

# Data Preparation for Visualization

In [ ]:
df_clean = ewaste_2012_2023.rename(columns={
    'Geopolitical entity (reporting)': 'Country',
    'Time': 'Year',
    'Waste management operations': 'Operation',
    'Waste categories': 'Waste_Category',
    'value': 'Value',
    'Unit of measure': 'Unit'
})

In [ ]:
df_clean['Operation'].unique()

In [ ]:
# Define Operation Constants 
GEN = 'Waste generated' 
COL = 'Waste collected'
REC = 'Waste recovery'
TRE = 'Waste treatment'
RECY = 'Waste Recycling'


In [ ]:
# Pivot the data
# We aggregate by Country and Year to get totals across all waste categories first
df_pivot = df_clean.pivot_table(
    index=['Country', 'Year'],
    columns='Operation',
    values='Value',
    aggfunc='sum'
).reset_index()

## Calculate Key Performance Indicators (KPIs)

In [ ]:
# CALCULATE CIRCULARITY INDEX (RQ8)
# Formula: Recovery Rate = Recovered / Generated

ops = df_clean['Operation'].unique()

# Dynamically find the exact column names that match our keywords
gen_col = [x for x in ops if 'generated' in x.lower()][0]
rec_col = [x for x in ops if 'recovery' in x.lower()][0]
col_col = [x for x in ops if 'collected' in x.lower()][0]
tre_col = [x for x in ops if 'treatment' in x.lower()][0]
recy_col = [x for x in ops if 'recycling' in x.lower()][0]

df_pivot['Circularity_Index'] = df_pivot[rec_col] / df_pivot[gen_col]
df_pivot['Collection_Efficiency'] = df_pivot[col_col] / df_pivot[gen_col]
df_pivot = df_pivot.fillna(0) # Fill NaN values with 0 to prevent errors during plotting

print("✅ Data Prep Complete. Columns renamed & Metrics calculated.")


# Exploratory Data Analysis


## Trends & Policy Impact
Visualizes how efficiency changed over time and the impact of the 2018 directive.
This visualizes how the circularity index (Recovery / Generation) has changed from 2012 to 2023.

In [ ]:
# Trend Lines by Country
fig1 = px.line(df_pivot, x='Year', y='Circularity_Index', color='Country',
              title='RQ1: E-Waste Circularity Index Trends (2012-2023)',
              labels={'Circularity_Index': 'Recovery Rate (Recovered/Generated)'},
              markers=True)
fig1.show()

## Pre vs Post 2018 Analysis (Impact of Policy Changes) 
We compare the distribution of the Circularity Index before and after 2018 to see if the new directive made a statistical difference.

In [ ]:
df_pivot['Period'] = np.where(df_pivot['Year'] < 2018, 'Pre-2018', 'Post-2018')

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_pivot, x='Period', y='Circularity_Index', palette='Set2') #Plot a Boxplot to visualize the spread and median of the data.
plt.title("RQ6: Impact of 2018 Open Scope Directive on Circularity")
plt.ylabel("Circularity Index")
plt.show()

## Waste Composition Changes (Stacked Area Chart)
This shows which types of e-waste (categories) are growing the fastest.


In [ ]:
# Filtering for only "Waste Generated" operations and aggregate by Year and Category.
df_cats = df_clean[df_clean['Operation'] == gen_col].groupby(['Year', 'Waste_Category'])['Value'].sum().reset_index()

fig9 = px.area(df_cats, x='Year', y='Value', color='Waste_Category',
              title='RQ9: Evolution of E-Waste Composition (Total Generation)',
              groupnorm='percent') # Show as percentage to see composition shift
fig9.show()

## Choropleth Map (Geographic Distribution)
Visualizes the average performance of countries on a map for the most recent years (2021+)

In [ ]:
# Choropleth Map (2022/2023 average)
# calculate the average index per country for recent data
df_recent = df_pivot[df_pivot['Year'] >= 2021].groupby('Country')[['Circularity_Index']].mean().reset_index()

#Create the map. 'locations' matches country names
fig2 = px.choropleth(df_recent, 
                     locations="Country", 
                     locationmode="country names",
                     color="Circularity_Index", 
                     hover_name="Country",
                     color_continuous_scale="Viridis",
                     title="RQ2: European Circularity Intensity Map (Recent Avg)",
                     scope="europe")
fig2.show()

## Identifying Waste Hotspots (Risk Zones Bubble Chart) 
A bubble chart to show three dimensions: 
 X=Generation, Y=Collection Efficiency, Size=Gap (The "Risk")

In [ ]:
# Calculate the 'Waste Gap' (Uncollected Waste)
# We use .clip(lower=0) to ensure we don't pass negative numbers to the plot
df_pivot['Waste_Gap'] = (df_pivot[gen_col] - df_pivot[rec_col]).clip(lower=0)

target_year = 2021  
df_risk = df_pivot[df_pivot['Year'] == target_year].copy()

# Generate the Bubble Chart
fig7 = px.scatter(
    df_risk, 
    x=gen_col, 
    y='Circularity_Index',
    size='Waste_Gap', # Larger bubble = More uncollected waste (High Risk)
    color='Country',
    hover_name='Country',
    title=f"RQ7: Waste Hotspots ({target_year}) - Size = Unrecovered Waste Gap",
    labels={
        gen_col: "Total Waste Generated (Tonnes)", 
        'Circularity_Index': "Recovery Rate (0-1)",
        'Waste_Gap': 'Unrecovered Waste (Tonnes)'
    },
    size_max=60  # Adjusts the scaling of the largest bubbles
)

# Add Threshold Lines 
# Vertical line: Average Generation
fig7.add_vline(x=df_risk[gen_col].mean(), line_width=1, line_dash="dash", line_color="grey")
# Horizontal line: 50% Recovery Target
fig7.add_hline(y=0.5, line_width=1, line_dash="dash", line_color="red", annotation_text="Target")

fig7.show()

# Advanced Visualization (Sankey & Heatmap)
This section handles complex flows and data quality checks.

## Sankey Diagram Logic (Visualizing Flows) 
A Sankey diagram visualizes the flow of mass from Generation -> Collection -> Recovery.

In [ ]:
import plotly.graph_objects as go

# Aggregating total flows for 2021
df_sankey = df_pivot[df_pivot['Year'] == 2021]
total_gen = df_sankey[gen_col].sum()
total_col = df_sankey[col_col].sum()
total_rec = df_sankey[rec_col].sum()

# Define Links
# Source 0 (Gen) -> Target 1 (Col)
# Source 0 (Gen) -> Target 2 (Uncollected)
# Source 1 (Col) -> Target 3 (Recovered)
# Source 1 (Col) -> Target 4 (Treatment Loss)

fig3 = go.Figure(data=[go.Sankey(
    node = dict(
        pad = 15, thickness = 20, line = dict(color = "black", width = 0.5),
        label = ["Generated", "Collected", "Uncollected", "Recovered", "Treatment Loss"],
        color = "blue"
    ),
    link = dict(
        source = [0, 0, 1, 1], 
        target = [1, 2, 3, 4],
        value = [total_col, (total_gen - total_col), total_rec, (total_col - total_rec)]
    ))])

fig3.update_layout(title_text="RQ3: Waste Flow Analysis (2021)", font_size=10)
fig3.show()

## Data Quality 
Novel analysis to flag reporting gaps.

In [ ]:
#Missing Data Heatmap
# Pivot raw data to see if specific Operations are missing for specific Countries/Years
data_completeness = df_clean.pivot_table(index='Country', columns='Year', values='Value', aggfunc='count')

plt.figure(figsize=(12, 8))
sns.heatmap(data_completeness.isnull(), cbar=False, cmap='Reds')
plt.title('RQ5: Data Transparency Map (Red = Missing Data)')
plt.xlabel('Year')
plt.ylabel('Country')
plt.show()

# Calculate Completeness Score
completeness_score = data_completeness.notna().mean(axis=1).sort_values()
print("Top 5 Countries with Missing Data Issues:\n", completeness_score.head())

## Machine Learning (Clustering Archetypes)
This section uses K-Means to group countries based on their performance, rather than just geography.

In [ ]:
# Prepare Features
# Aggregate by Country Mean over all years
features = df_pivot.groupby('Country')[[gen_col, 'Circularity_Index']].mean()

# Data Cleaning: Replace Inf with NaN, then Drop
features = features.replace([np.inf, -np.inf], np.nan).dropna()

# CLUSTERING LOGIC
n_samples = len(features)
print(f"ℹ️ Valid Countries for Clustering: {n_samples}")

if n_samples < 2:
    print("⚠️ Not enough data points to perform clustering (need at least 2).")
else:
    # Dynamically set k to be at most n_samples, but prefer 3
    k = min(3, n_samples)
    print(f"🔄 Adapting strategy: Using k={k} clusters.")

In [ ]:
# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# Apply K-Means
kmeans = KMeans(n_clusters=k, random_state=42)
features['Cluster'] = kmeans.fit_predict(X_scaled)

# Visualize cluster
fig10 = px.scatter(
    features.reset_index(), 
    x=gen_col, 
    y='Circularity_Index', 
    color=features['Cluster'].astype(str),
    hover_name='Country',
    title=f"RQ10: Country Archetypes ({k} Clusters)",
    labels={
        gen_col: 'Avg Generation (Tonnes)', 
        'Circularity_Index': 'Avg Recovery Rate',
        'color': 'Cluster'
},
       
    symbol=features['Cluster'].astype(str)
)
fig10.show()

# Print Cluster Characteristics
print("\nCluster Averages:")
print(features.groupby('Cluster').mean())

## Top 5 E-Waste Generating Countries (2018-2023)
This final section refines the analysis to focus on the largest contributors to E-waste.

In [ ]:
# Re-create a clean dataframe to ensure no previous filters interfere
df = ewaste_2012_2023.rename(columns={
    'Geopolitical entity (reporting)': 'Country',
    'Time': 'Year',
    'Waste management operations': 'Operation',
    'Waste categories': 'Waste_Category',
    'value': 'Value',
    'Unit of measure': 'Unit'
})

# Filter for 2018-2023 and Unit = Tonne
df = df[(df['Year'] >= 2018) & (df['Year'] <= 2023) & (df['Unit'] == 'Tonne')]

In [ ]:
# Find the "Waste generated" operation name
    
gen_ops = [op for op in df['Operation'].unique() if 'generated' in op.lower()]
if not gen_ops:
     print("Could not find 'Waste generated' operation. Available operations:", df['Operation'].unique())
else:
    gen_op = gen_ops[0] # Assume the first one is correct, 'Waste generated'
# We sum the 'Waste generated' for all years and pick the top 5      
    top_5_countries = df[df['Operation'] == gen_op].groupby('Country')['Value'].sum().nlargest(5).index.tolist()
    print(f"Top 5 E-Waste Generating Countries (2018-2023): {top_5_countries}")

In [ ]:
# Filter for Top 5 Countries and Management Operations
# We want to compare Recycling vs Recovery vs Disposal vs Collection
        
target_keywords = ['recycling', 'recovery', 'disposal', 'prepare for reuse', 'reuse']
#Re-create a clean dataframe to ensure no previous filters interfere.
        
# Apply the mapping
def map_operation(op_name):
        op_lower = op_name.lower()
        if 'recycling' in op_lower: return 'Recycling'
        if 'recovery' in op_lower and 'recycling' not in op_lower: return 'Other Recovery' 
        if 'disposal' in op_lower: return 'Disposal'
        if 'reuse' in op_lower: return 'Reuse'
        return None

df['Op_Category'] = df['Operation'].apply(map_operation)
        
# Filter for the top 5 countries and the mapped categories
df_top5 = df[df['Country'].isin(top_5_countries) & df['Op_Category'].notna()]

# Sum values by Country and Category
df_agg = df_top5.groupby(['Country', 'Op_Category'])['Value'].sum().reset_index()
        
# Calculate total treated/managed per country to get percentages
df_totals = df_agg.groupby('Country')['Value'].sum().reset_index().rename(columns={'Value': 'Total_Managed'})
df_agg = df_agg.merge(df_totals, on='Country')
df_agg['Proportion'] = df_agg['Value'] / df_agg['Total_Managed']
        
# Visualize
fig = px.bar(df_agg, 
            x='Country', 
            y='Proportion', 
            color='Op_Category',
            title='Proportional Breakdown of Waste Management Operations (Top 5 Generating Countries, 2018-2023)',
            text_auto='.1%',
            labels={'Proportion': 'Share of Managed Waste', 'Op_Category': 'Operation'},
            category_orders={"Country": top_5_countries} # Order by generation rank
            )
fig.show()

In [ ]:
START_YEAR = 2012
END_YEAR = 2023

df_chart = ewaste_2012_2023.copy()
df_chart = df_chart.rename(columns={
    'Time': 'Year', 
    'Waste management operations': 'Operation', 
    'value': 'Value',
    'Geopolitical entity (reporting)': 'Country'
})

# Filter by Year
df_chart = df_chart[(df_chart['Year'] >= START_YEAR) & (df_chart['Year'] <= END_YEAR)]

# Filter for key Operations (Generation, Collection, Recycling)

target_ops = [
    'Waste generated', 
    'Products put on the market', 
    'Waste collected', 
    'Recycling', 
    'Recovery', 
    'Waste recovered'
]

# Keep only rows that match our target operations
df_chart = df_chart[df_chart['Operation'].str.contains('|'.join(target_ops), case=False, na=False)]

# Standardize Operation Names for the Legend
def standardize_op(x):
    if 'generated' in x.lower() or 'market' in x.lower(): return 'Generation'
    if 'collected' in x.lower(): return 'Collection'
    if 'recycling' in x.lower() or 'recovered' in x.lower(): return 'Recycling/Recovery'
    return x

df_chart['Operation_Label'] = df_chart['Operation'].apply(standardize_op)

# Aggregate (Sum values across all countries to show Total Performance)

if 'Unit of measure' in ewaste_2012_2023.columns:
    df_chart = df_chart[df_chart['Unit of measure'] == 'Tonne']

df_grouped = df_chart.groupby(['Year', 'Operation_Label'])['Value'].sum().reset_index()

In [ ]:
fig = px.bar(
    df_grouped, 
    x='Year', 
    y='Value', 
    color='Operation_Label',
    barmode='group', # Groups bars side-by-side for comparison
    title=f'E-Waste Performance: Generation vs Collection vs Recycling ({START_YEAR}-{END_YEAR})',
    labels={'Value': 'Total Mass (Tonnes)', 'Operation_Label': 'Metric'},
    color_discrete_map={
        'Generation': '#EF553B',         # Red
        'Collection': '#636EFA',         # Blue
        'Recycling/Recovery': '#00CC96'  # Green
    }
)

fig.update_layout(xaxis=dict(tickmode='linear'), template='plotly_white')
fig.show()

In [ ]:
df_analysis = ewaste_2012_2023.copy()

# Rename columns for ease of use
df_analysis = df_analysis.rename(columns={
    'Geopolitical entity (reporting)': 'Country',
    'Time': 'Year',
    'Waste management operations': 'Operation',
    'value': 'Value',
    'Unit of measure': 'Unit'
})

# Filter for the relevant years and unit
df_analysis = df_analysis[
    (df_analysis['Year'] >= 2018) & 
    (df_analysis['Year'] <= 2023) & 
    (df_analysis['Unit'] == 'Tonne')
]

# IDENTIFY TOP 5 GENERATORS

# Find the exact 'Generation' operation name in your data

gen_ops = [op for op in df_analysis['Operation'].unique() if 'generated' in op.lower()]
gen_op = gen_ops[0] if gen_ops else 'Waste generated'

# Aggregate Generation by Country
top_5_countries = (
    df_analysis[df_analysis['Operation'] == gen_op]
    .groupby('Country')['Value']
    .sum()
    .nlargest(5)
    .index.tolist()
)

print(f"🌍 Top 5 Generating Countries: {top_5_countries}")


# DEFINE MANAGEMENT CATEGORIES

op_map = {
    'Waste recycled': 'Recycling',
    'Recycling': 'Recycling',
    'Preparing for reuse': 'Reuse',
    'Energy recovery': 'Recovery (Energy)',
    'Recovery': 'Recovery (Total)', # Be careful not to double count if using specific types
    'Disposal': 'Disposal',
    'Landfill': 'Disposal',
    'Waste collected': 'Collection'
}

# Create a simplified 'Category' column
def categorize_op(x):
    for key, val in op_map.items():
        if key.lower() in x.lower():
            return val
    return None

df_analysis['Category'] = df_analysis['Operation'].apply(categorize_op)

# Filter for Top 5 Countries and Valid Categories
df_top5 = df_analysis[
    (df_analysis['Country'].isin(top_5_countries)) & 
    (df_analysis['Category'].notna())
]


# CALCULATE AGGREGATE TOTALS (2018-2023)


df_breakdown = df_top5.groupby(['Country', 'Category'])['Value'].sum().reset_index()


# GENERATE VISUALIZATIONS

# Collection vs Generation (The Gap)

df_gap = df_breakdown[df_breakdown['Category'].isin(['Recycling', 'Recovery (Energy)', 'Disposal', 'Collection'])]

# Treatment Breakdown (Recycling vs Recovery vs Disposal)
# We filter only the "End of Life" treatments to show the split
treatment_cats = ['Recycling', 'Recovery (Energy)', 'Disposal']
df_treatment = df_breakdown[df_breakdown['Category'].isin(treatment_cats)].copy()

# Calculate percentages
df_totals = df_treatment.groupby('Country')['Value'].transform('sum')
df_treatment['Percentage'] = (df_treatment['Value'] / df_totals) * 100

fig = px.bar(
    df_treatment, 
    x='Country', 
    y='Percentage', 
    color='Category',
    title=f'Proportional Breakdown of Waste Treatment (Top 5 Generators, 2018-2023)',
    text_auto='.1f',
    labels={'Percentage': 'Share of Treated Waste (%)'},
    category_orders={'Country': top_5_countries},
    color_discrete_map={
        'Recycling': '#2ca02c',  # Green
        'Recovery (Energy)': '#ff7f0e', # Orange
        'Disposal': '#d62728'    # Red
    }
)

fig.update_layout(yaxis_title="Percentage of Treated Waste", barmode='stack')
fig.show()

# Print the Raw Data for your Report
print("\n📊 Raw Volumes (Tonnes) by Operation:")
print(df_breakdown.pivot(index='Country', columns='Category', values='Value'))

In [ ]:
# Update the Mapping Function to include Generation & Collection
def map_operation(op_name):
    op_lower = op_name.lower()
    
    # Priority matching (specific to general)
    if 'generated' in op_lower or 'put on the market' in op_lower:
        return 'Generation'
    if 'collected' in op_lower:
        return 'Collection'
    if 'recycling' in op_lower:
        return 'Recycling'
    if 'recovery' in op_lower and 'recycling' not in op_lower and 'energy' not in op_lower:
        return 'Recovery (Total)' # Capture broad recovery if distinct from energy
    if 'energy' in op_lower:
         return 'Energy Recovery'
    if 'disposal' in op_lower or 'landfill' in op_lower:
        return 'Disposal'
    if 'reuse' in op_lower:
        return 'Reuse'
    
    return None 

# Apply the mapping
df['Op_Category'] = df['Operation'].apply(map_operation)

#Filter for Top 5 Countries and Valid Categories

df_top5 = df[df['Country'].isin(top_5_countries) & df['Op_Category'].notna()].copy()

#  Aggregate Values
df_agg = df_top5.groupby(['Country', 'Op_Category'])['Value'].sum().reset_index()

# Calculate Proportions Relative to GENERATION
# Find the total generation for each country to use as the baseline (100%)
df_gen = df_agg[df_agg['Op_Category'] == 'Generation'][['Country', 'Value']].rename(columns={'Value': 'Total_Generated'})

# Merge generation totals back into the main dataframe
df_agg = df_agg.merge(df_gen, on='Country', how='left')

# Calculate percentage (Value / Total Generated)
# For 'Generation', this will be 100%. For 'Collection', this is the Collection Rate.
df_agg['Proportion'] = df_agg['Value'] / df_agg['Total_Generated']

# Visualize
# We use 'group' mode because these categories overlap 
fig = px.bar(
    df_agg, 
    x='Country', 
    y='Proportion', 
    color='Op_Category',
    barmode='group', # compares them side-by-side
    title='Comparison of Waste Operations vs Generation (2018-2023)',
    text_auto='.1%',
    labels={'Proportion': 'Percentage of Waste Generated', 'Op_Category': 'Operation'},
    category_orders={
        "Country": top_5_countries,
        "Op_Category": ["Generation", "Collection", "Recycling", "Energy Recovery", "Disposal"]
    }
)

# Add a reference line for 100%
fig.add_hline(y=1.0, line_dash="dot", annotation_text="Generation Baseline (100%)")

fig.show()